# 🔍 Revue de Code Terraform

Notebook pour effectuer des revues complètes de code Terraform.

**Fonctionnalités:**
- 🔒 Audit de sécurité (secrets, permissions, chiffrement)
- 📋 Conformité aux standards (nommage, variables, documentation)
- 🎨 Qualité du code (style, clarté, optimisations)
- 📚 Base de connaissances des best practices Terraform

**Workflow:**
1. Initialiser les composants (une fois)
2. Lancer une revue avec `reviewer.review(path)`
3. Consulter le rapport détaillé

**Modèles:**
- Revue: Qwen2.5-coder (Ollama)
- Embeddings: nomic-embed-text (Ollama)

## 1️⃣ Initialisation

In [1]:
# Configuration de base
import sys
import logging
from pathlib import Path
from dotenv import load_dotenv

# Charger les variables d'environnement
load_dotenv()

# Ajouter le projet au path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)

print("✅ Configuration de base chargée")

✅ Configuration de base chargée


In [2]:
# Imports des modules du projet
from terraform_agent import (
    Config,
    KnowledgeBase,
    PromptManager,
    TerraformReviewer
)

print("✅ Modules importés")

14:45:20 - INFO - 📋 Ensuring phoenix working directory: /Users/melkouhen/.phoenix
14:45:21 - INFO - setup plugin alembic.autogenerate.schemas
14:45:21 - INFO - setup plugin alembic.autogenerate.tables
14:45:21 - INFO - setup plugin alembic.autogenerate.types
14:45:21 - INFO - setup plugin alembic.autogenerate.constraints
14:45:21 - INFO - setup plugin alembic.autogenerate.defaults
14:45:21 - INFO - setup plugin alembic.autogenerate.comments


✅ Modules importés


In [3]:
# Initialisation des composants
print("⏳ Initialisation en cours...\n")

# Configuration
config = Config(base_dir=project_root)
print(f"✓ Configuration")
print(f"  - Project root: {config.PROJECT_ROOT}")
print(f"  - Work dir: {config.WORK_DIR}")
print(f"  - Review model: {config.REVIEW_MODEL_NAME}\n")

# Base de connaissances
knowledge_base = KnowledgeBase(config=config)
print("\n✓ Base de connaissances indexée\n")

# Prompts
prompts = PromptManager(config=config)
print("✓ Prompts chargés\n")

# Reviewer
reviewer = TerraformReviewer(
    config=config,
    knowledge_base=knowledge_base,
    prompts=prompts
)
print("✓ Reviewer initialisé\n")

print("="*60)
print("🎉 Tous les composants sont prêts !")
print("="*60)

14:45:23 - INFO - Clearing 232 existing documents from vectorstore


⏳ Initialisation en cours...

✓ Configuration
  - Project root: /Users/melkouhen/audit-tools/test-langchain
  - Work dir: /Users/melkouhen/audit-tools/test-langchain/work
  - Review model: qwen2.5-coder:7b-instruct

📚 Loading knowledge base...
  ✓ Loaded 29 document(s) from /Users/melkouhen/audit-tools/test-langchain/rules
  ✓ Split into 232 chunks
  Creating new vectorstore database...
  🗑️ Clearing 232 existing documents...


KeyboardInterrupt: 

## 2️⃣ Revue de Code

### Exemple 1: Revue Standard

In [ ]:
# Chemin du code à analyser
code_path = str(config.WORK_DIR / "envs/dev")

# Vérifier que le chemin existe
if Path(code_path).exists():
    tf_files = list(Path(code_path).glob("*.tf"))
    print(f"✅ Chemin trouvé: {code_path}")
    print(f"📄 Fichiers: {len(tf_files)} fichiers .tf")
    for f in tf_files:
        size_kb = f.stat().st_size / 1024
        print(f"   - {f.name} ({size_kb:.1f} KB)")
else:
    print(f"⚠️  Chemin introuvable: {code_path}")
    print(f"\nChemins disponibles:")
    for item in config.WORK_DIR.rglob("*.tf"):
        print(f"   - {item.parent}")

In [ ]:
# Lancer la revue complète
report = reviewer.review(code_path, verbose=True)

# Afficher le rapport
print("\n" + "="*80)
print("📋 RAPPORT DE REVUE")
print("="*80 + "\n")
print(report)

### Exemple 2: Revue Ciblée (Sécurité)

In [ ]:
# Revue focalisée sur la sécurité
security_report = reviewer.review(
    path=code_path,
    query="terraform security best practices secrets encryption IAM",
    verbose=True
)

print("\n" + "="*80)
print("🔒 AUDIT DE SÉCURITÉ")
print("="*80 + "\n")
print(security_report)

### Exemple 3: Revue Ciblée (Conformité)

In [ ]:
# Revue focalisée sur la conformité
compliance_report = reviewer.review(
    path=code_path,
    query="terraform naming conventions variables outputs documentation",
    verbose=True
)

print("\n" + "="*80)
print("📋 AUDIT DE CONFORMITÉ")
print("="*80 + "\n")
print(compliance_report)

## 3️⃣ Utilitaires

### Scan Rapide (sans revue complète)

In [ ]:
# Obtenir des statistiques rapides
stats = reviewer.quick_scan(code_path)

print("📊 Statistiques du Code\n")
print(f"Chemin: {stats['path']}")
print(f"Fichiers .tf: {stats['files']}")
print(f"Ressources: {stats['resources']}")
print(f"Variables: {stats['variables']}")
print(f"Outputs: {stats['outputs']}")
print(f"Lignes totales: {stats['total_lines']}")

### Trouver tous les répertoires avec du code Terraform

In [ ]:
# Lister tous les répertoires avec des fichiers .tf
def find_terraform_dirs():
    """Find all directories containing .tf files."""
    directories = set()
    for tf_file in config.WORK_DIR.rglob("*.tf"):
        directories.add(tf_file.parent)
    return sorted(directories)

terraform_dirs = find_terraform_dirs()

print(f"📁 Répertoires avec du code Terraform: {len(terraform_dirs)}\n")
for directory in terraform_dirs:
    tf_count = len(list(directory.glob("*.tf")))
    rel_path = directory.relative_to(config.WORK_DIR)
    print(f"  - {rel_path} ({tf_count} fichiers)")

### Comparer plusieurs revues

In [ ]:
def compare_reviews(path: str, queries: list[str]) -> dict:
    """
    Compare multiple reviews with different focus areas.
    
    Args:
        path: Path to Terraform code
        queries: List of queries for different aspects
        
    Returns:
        Dictionary mapping queries to review results
    """
    results = {}
    
    for i, query in enumerate(queries, 1):
        print(f"\n{'='*80}")
        print(f"Revue {i}/{len(queries)}: {query}")
        print("="*80)
        
        result = reviewer.review(path, query, verbose=False)
        results[query] = result
        
        # Analyser le résultat
        has_critical = "CRITIQUE" in result
        has_major = "MAJEUR" in result
        
        if has_critical:
            print("❌ Problèmes CRITIQUES détectés")
        elif has_major:
            print("⚠️  Problèmes MAJEURS détectés")
        else:
            print("✅ Code conforme")
    
    return results

# Exemple d'utilisation:
# queries = [
#     "terraform security best practices",
#     "terraform naming conventions",
#     "terraform documentation"
# ]
# comparison = compare_reviews(code_path, queries)

## 4️⃣ Sauvegarde des Rapports

In [ ]:
from datetime import datetime

def save_report(report: str, name: str = None, output_dir: Path = None) -> Path:
    """
    Save review report to file.
    
    Args:
        report: Review report content
        name: Custom name for report (optional)
        output_dir: Output directory (default: work/)
        
    Returns:
        Path to saved file
    """
    if output_dir is None:
        output_dir = config.WORK_DIR
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if name:
        filename = f"review_{name}_{timestamp}.md"
    else:
        filename = f"review_report_{timestamp}.md"
    
    filepath = output_dir / filename
    
    header = f"""# Rapport de Revue de Code Terraform

**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Modèle:** {config.REVIEW_MODEL_NAME}
**Généré par:** TerraformReviewer

---

"""
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(header + report)
    
    size_kb = filepath.stat().st_size / 1024
    print(f"✅ Rapport sauvegardé: {filepath}")
    print(f"   Taille: {size_kb:.1f} KB")
    
    return filepath

# Exemples d'utilisation:
# save_report(report, name="standard")
# save_report(security_report, name="security")
# save_report(compliance_report, name="compliance")

## 5️⃣ Revue avec Chemin Personnalisé

In [ ]:
# Pour analyser un répertoire personnalisé
custom_path = "/chemin/vers/votre/code/terraform"

# Vérifier et scanner
# if Path(custom_path).exists():
#     stats = reviewer.quick_scan(custom_path)
#     print(f"Fichiers: {stats['files']}, Ressources: {stats['resources']}")
#     
#     # Lancer la revue
#     custom_report = reviewer.review(custom_path)
#     print(custom_report)
#     
#     # Sauvegarder
#     save_report(custom_report, name="custom")
# else:
#     print(f"Chemin introuvable: {custom_path}")

---

## 📚 Guide d'Utilisation

### Workflow Standard

```python
# 1. Initialiser (une fois)
# Exécuter les cellules 1-3

# 2. Revue simple
report = reviewer.review("/path/to/terraform")
print(report)

# 3. Sauvegarder
save_report(report)
```

### Types de Revues

**Standard** - Best practices générales
```python
reviewer.review(path)
```

**Sécurité** - Audit de sécurité approfondi
```python
reviewer.review(path, query="terraform security secrets encryption IAM")
```

**Conformité** - Standards et conventions
```python
reviewer.review(path, query="terraform naming conventions documentation")
```

**Performance** - Optimisations et coûts
```python
reviewer.review(path, query="terraform performance optimization cost")
```

### Outils Disponibles

- `reviewer.review(path, query, verbose)` - Revue complète
- `reviewer.quick_scan(path)` - Statistiques rapides
- `save_report(report, name)` - Sauvegarde avec timestamp
- `compare_reviews(path, queries)` - Comparaison multi-aspects

### Classification des Problèmes

🔴 **CRITIQUE** - Sécurité, runtime, perte de données (bloquant)
🟠 **MAJEUR** - Maintenabilité, performance, best practices
🟡 **MINEUR** - Style, optimisations optionnelles

### Troubleshooting

**Ollama non lancé**
```bash
ollama serve
```

**Modèle manquant**
```bash
ollama pull qwen2.5-coder:7b-instruct
ollama pull nomic-embed-text
```

**Logs détaillés**
```python
import logging
logging.getLogger('terraform_agent').setLevel(logging.DEBUG)
```